In [1]:
import os
import pandas as pd
import numpy as np
import psutil
import random

SEED = 32
np.random.seed(SEED)


from itertools import product

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from utils.utils import load_data, preprocess_data, measure_duration

In [2]:
# Load the datasets
df_maternal = load_data('../data/raw/uci/maternal_health_risk/maternal_health_risk.csv')
df_amazon = load_data('../data/raw/kaggle/reviews/amazon_review_ID.shuf.lrn.csv')

Data successfully loaded from ../data/raw/uci/maternal_health_risk/maternal_health_risk.csv. First 5 rows:
   Age  SystolicBP  DiastolicBP    BS  BodyTemp  HeartRate  RiskLevel
0   25         130           80  15.0      98.0         86  high risk
1   35         140           90  13.0      98.0         70  high risk
2   29          90           70   8.0     100.0         80  high risk
3   30         140           85   7.0      98.0         70  high risk
4   35         120           60   6.1      98.0         76   low risk
Data successfully loaded from ../data/raw/kaggle/reviews/amazon_review_ID.shuf.lrn.csv. First 5 rows:
   ID  V1  V2  V3  V4  V5  V6  V7  V8  V9  ...  V9992  V9993  V9994  V9995  \
0   0  17   4   8   8   9   4   0   2   3  ...      0      0      0      0   
1   1  21   9   5   8   6   2  16   3  12  ...      0      0      0      2   
2   2   9   7   6   3   8   2   9   4   4  ...      0      0      0      0   
3   3   8   3   5   2   4   3   8   2   4  ...      0      

In [3]:
# Example for Dataset 1:
if df_maternal is not None:
    X1, y1 = preprocess_data(df_maternal, label_col="RiskLevel")

# Example for Dataset 2:
if df_amazon is not None:
    X2, y2 = preprocess_data(df_amazon, label_col="Class")

Preprocessing complete: 1014 samples, 6 features, 3 classes.
Preprocessing complete: 750 samples, 10001 features, 50 classes.


In [4]:
# 1) Train/test split for dataset 1
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)

# 2) One-hot encoding for labels of dataset 1
ohe1 = OneHotEncoder()
y1_train_ohe = ohe1.fit_transform(y1_train.reshape(-1, 1)).toarray()
y1_test_ohe  = ohe1.transform(y1_test.reshape(-1, 1)).toarray()

# Output the shapes of the datasets
print("Dataset 1 shapes:")
print(f"  X1_train: {X1_train.shape}, y1_train_ohe: {y1_train_ohe.shape}")
print(f"  X1_test : {X1_test.shape},  y1_test_ohe : {y1_test_ohe.shape}")

# 3) Train/test split for dataset 2
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

# 4) One-hot encoding for labels of dataset 2
ohe2 = OneHotEncoder()
y2_train_ohe = ohe2.fit_transform(y2_train.reshape(-1, 1)).toarray()
y2_test_ohe  = ohe2.transform(y2_test.reshape(-1, 1)).toarray()

# Output the shapes of the datasets
print("\nDataset 2 shapes:")
print(f"  X2_train: {X2_train.shape}, y2_train_ohe: {y2_train_ohe.shape}")
print(f"  X2_test : {X2_test.shape},  y2_test_ohe : {y2_test_ohe.shape}")

Dataset 1 shapes:
  X1_train: (811, 6), y1_train_ohe: (811, 3)
  X1_test : (203, 6),  y1_test_ohe : (203, 3)

Dataset 2 shapes:
  X2_train: (600, 10001), y2_train_ohe: (600, 50)
  X2_test : (150, 10001),  y2_test_ohe : (150, 50)


In [5]:
class Sigmoid:
    @staticmethod
    def forward(x):
        return 1 / (1 + np.exp(-x))

    @staticmethod
    def backward(x):
        sig = Sigmoid.forward(x)
        return sig * (1 - sig)
    

class ReLU:
    @staticmethod
    def forward(x):
        return np.maximum(0, x)

    @staticmethod
    def backward(x):
        return (x > 0).astype(float)
    

class Tanh:
    @staticmethod
    def forward(x):
        return np.tanh(x)

    @staticmethod
    def backward(x):
        return 1 - np.tanh(x) ** 2

In [6]:
class NeuralNetwork:
    def __init__(self, input_size: int, hidden_layers: int, hidden_units: int,
                 output_size: int, activation: str = "sigmoid", seed: int = 32):
        
        np.random.seed(seed)
        
        ops = {
            "sigmoid": (Sigmoid.forward, Sigmoid.backward),
            "relu":    (ReLU.forward,    ReLU.backward),
            "tanh":    (Tanh.forward,    Tanh.backward),
        }
        self.act_forward, self.act_backward = ops[activation]
        # Layer sizes
        self.layer_sizes = [input_size] + [hidden_units]*hidden_layers + [output_size]
        # Initialize weights & biases
        self.weights = [np.random.randn(self.layer_sizes[i], self.layer_sizes[i+1]) * 0.1
                        for i in range(len(self.layer_sizes)-1)]
        self.biases  = [np.zeros((1, self.layer_sizes[i+1]))
                        for i in range(len(self.layer_sizes)-1)]
    def forward(self, X):
        X = np.asarray(X)
        self.z_values = []
        self.a_values = [X]
        a = X
        for W,b in zip(self.weights, self.biases):
            z = a.dot(W) + b
            self.z_values.append(z)
            a = self.act_forward(z)
            self.a_values.append(a)
        return a
    def backward(self, y_true, learning_rate):
        y_true = np.asarray(y_true)
        m = y_true.shape[0]
        # Compute output layer delta (MSE derivative)
        delta = (self.a_values[-1] - y_true) * self.act_backward(self.z_values[-1])
        for i in reversed(range(len(self.weights))):
            a_prev = self.a_values[i]
            dW = a_prev.T.dot(delta) / m
            db = delta.sum(axis=0, keepdims=True) / m
            self.weights[i] -= learning_rate * dW
            self.biases[i]  -= learning_rate * db
            if i>0:
                delta = delta.dot(self.weights[i].T) * self.act_backward(self.z_values[i-1])
    def train(self, X, y, epochs, learning_rate, verbose=False):
        X = np.asarray(X)
        y = np.asarray(y)
        for epoch in range(1, epochs+1):
            self.forward(X)
            self.backward(y, learning_rate)
            if verbose and epoch%100==0:
                loss = np.mean((y - self.a_values[-1])**2)
                print(f"Epoch {epoch}/{epochs} — Loss: {loss:.4f}")
    def predict(self, X):
        X = np.asarray(X)
        probs = self.forward(X)
        return np.argmax(probs, axis=1)
    def count_parameters(self):
        return sum(W.size + b.size for W,b in zip(self.weights, self.biases))
    def compute_ram_usage(self):
        process = psutil.Process(os.getpid())
        mem_bytes = process.memory_info().rss  # Resident Set Size
        return mem_bytes / (1024 ** 2)


In [ ]:
activations_list = ['sigmoid', 'relu', 'tanh']
hidden_units_list = [8, 16, 32]

def run_experiments(X_train, y_train_ohe, X_test, y_test_ohe,
                    input_size, output_size, epochs=500, learning_rate=0.1, verbose=False):
    results = []
    y_test_labels = y_test_ohe.argmax(axis=1)
    for activation in activations_list:
        for hu in hidden_units_list:
            nn = NeuralNetwork(input_size, 1, hu, output_size, activation)
            nn.train(X_train, y_train_ohe, epochs, learning_rate, verbose)
            preds = nn.predict(X_test)
            acc = accuracy_score(y_test_labels, preds)
            f1 = f1_score(y_test_labels, preds, average='weighted')
            loss = ((y_test_ohe - nn.forward(X_test))**2).mean()
            results.append({
                'Activation': activation.capitalize(),
                'Hidden Layers': 1,
                'Hidden Units': hu,
                'Accuracy': f"{acc*100:.2f}%",
                'F1 Score': f"{f1:.4f}",
                'Final Loss': f"{loss:.4f}",
                'Parameters': nn.count_parameters(),
                'RAM Used (MB)': f"{nn.compute_ram_usage():.2f}"
            })
    return pd.DataFrame(results)

In [8]:
def grid_search(GRID, X_train, y_train_ohe, X_test, y_test_ohe, input_size, output_size, epochs=500, verbose=False):
    best, hist = None, []
    y_true = y_test_ohe.argmax(axis=1)
    for hl, hu, act, lr in product(GRID["hidden_layers"], GRID["hidden_units"], GRID["activation"], GRID["lr"]):
        nn = NeuralNetwork(input_size, hl, hu, output_size, act)
        nn.train(X_train, y_train_ohe, epochs, lr, verbose)
        preds = nn.predict(X_test)
        acc  = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds, average='weighted')
        loss = ((y_test_ohe - nn.forward(X_test))**2).mean()
        row  = {
            "Activation": act.capitalize(),
            "Hidden Layers": hl,
            "Hidden Units": hu,
            "Learning Rate": lr,
            "Accuracy": acc,
            "F1 Score": f1,
            "Final Loss": loss,
            "Parameters": nn.count_parameters(),
            "RAM MB": nn.compute_ram_usage()
        }
        hist.append(row)
        if best is None or acc > best["Accuracy"]:
            best = row
    return pd.DataFrame(hist), best

In [9]:
df1_results, s1, e1, dur1 = measure_duration(
    run_experiments,
    X_train=X1_train, y_train_ohe=y1_train_ohe,
    X_test=X1_test,  y_test_ohe=y1_test_ohe,
    input_size=X1_train.shape[1], output_size=y1_train_ohe.shape[1],
    epochs=1000, learning_rate=0.1, verbose=False
)


print(f"Implemntation Dataset 1 took {dur1}")

run_experiments start: 2025-05-26 22:38:28.569527
run_experiments   end: 2025-05-26 22:38:29.837486
run_experiments duration: 0:00:01.267959
Implemntation Dataset 1 took 0:00:01.267959


In [10]:
print(df1_results)

  Activation  Hidden Layers  Hidden Units Accuracy F1 Score Final Loss  \
0    Sigmoid              1             8   55.67%   0.4330     0.1888   
1    Sigmoid              1            16   55.17%   0.4277     0.1827   
2    Sigmoid              1            32   57.14%   0.4517     0.1757   
3       Relu              1             8   67.98%   0.6497     0.1463   
4       Relu              1            16   67.98%   0.6477     0.1406   
5       Relu              1            32   67.00%   0.6443     0.1440   
6       Tanh              1             8   66.50%   0.6381     0.1582   
7       Tanh              1            16   68.47%   0.6544     0.1570   
8       Tanh              1            32   67.98%   0.6512     0.1567   

   Parameters RAM Used (MB)  
0          83        415.64  
1         163        415.64  
2         323        415.64  
3          83        415.64  
4         163        415.64  
5         323        415.64  
6          83        415.64  
7         163      

In [11]:
df2_results, s3, e3, dur2 = measure_duration(
    run_experiments,
    X_train=X2_train, y_train_ohe=y2_train_ohe,
    X_test=X2_test,  y_test_ohe=y2_test_ohe,
    input_size=X2_train.shape[1], output_size=y2_train_ohe.shape[1],
    epochs=1000, learning_rate=0.1, verbose=True
)

print(f"Implemntation Dataset 2 took {dur2}")

Epoch 100/1000 — Loss: 0.0386
Epoch 200/1000 — Loss: 0.0276
Epoch 300/1000 — Loss: 0.0244
Epoch 400/1000 — Loss: 0.0230
Epoch 500/1000 — Loss: 0.0222
Epoch 600/1000 — Loss: 0.0217
Epoch 700/1000 — Loss: 0.0213
Epoch 800/1000 — Loss: 0.0211
Epoch 900/1000 — Loss: 0.0209
Epoch 1000/1000 — Loss: 0.0207
Epoch 100/1000 — Loss: 0.0297
Epoch 200/1000 — Loss: 0.0236
Epoch 300/1000 — Loss: 0.0219
Epoch 400/1000 — Loss: 0.0212
Epoch 500/1000 — Loss: 0.0208
Epoch 600/1000 — Loss: 0.0205
Epoch 700/1000 — Loss: 0.0203
Epoch 800/1000 — Loss: 0.0202
Epoch 900/1000 — Loss: 0.0201
Epoch 1000/1000 — Loss: 0.0200
Epoch 100/1000 — Loss: 0.0243
Epoch 200/1000 — Loss: 0.0212
Epoch 300/1000 — Loss: 0.0205
Epoch 400/1000 — Loss: 0.0201
Epoch 500/1000 — Loss: 0.0199
Epoch 600/1000 — Loss: 0.0198
Epoch 700/1000 — Loss: 0.0197
Epoch 800/1000 — Loss: 0.0197
Epoch 900/1000 — Loss: 0.0196
Epoch 1000/1000 — Loss: 0.0196
Epoch 100/1000 — Loss: 0.0200
Epoch 200/1000 — Loss: 0.0200
Epoch 300/1000 — Loss: 0.0200
Epoch 4

In [12]:
print(df2_results)

  Activation  Hidden Layers  Hidden Units Accuracy F1 Score Final Loss  \
0    Sigmoid              1             8    1.33%   0.0068     0.0272   
1    Sigmoid              1            16    0.67%   0.0036     0.0236   
2    Sigmoid              1            32    2.00%   0.0171     0.0212   
3       Relu              1             8    0.67%   0.0001     0.0200   
4       Relu              1            16    0.67%   0.0001     0.0200   
5       Relu              1            32    0.67%   0.0001     0.0251   
6       Tanh              1             8    2.00%   0.0091     0.0200   
7       Tanh              1            16    3.33%   0.0252     0.0206   
8       Tanh              1            32    4.67%   0.0497     0.0214   

   Parameters RAM Used (MB)  
0       80466        439.74  
1      160882        440.24  
2      321714        440.62  
3       80466        440.62  
4      160882        440.62  
5      321714        440.62  
6       80466        440.62  
7      160882      

In [13]:
grid_1 = {
    "hidden_layers": [1, 2, 3, 4, 5],
    "hidden_units":  [2, 4, 8, 16, 32, 64],
    "activation":    ["sigmoid", "relu", "tanh"],
    "lr":            [0.1, 0.05, 0.01]
}

(hist1, best1), s1, e1, dur3 = measure_duration(
    grid_search,
    GRID=grid_1,
    X_train=X1_train, y_train_ohe=y1_train_ohe,
    X_test=X1_test, y_test_ohe=y1_test_ohe,
    input_size=X1_train.shape[1],
    output_size=y1_train_ohe.shape[1],
    epochs=1000, verbose=False
)

print(f"Grid Search (Dataset 1) took {dur3}")


grid_search start: 2025-05-26 22:39:32.076538
grid_search   end: 2025-05-26 22:41:32.023018
grid_search duration: 0:01:59.946480
Grid Search (Dataset 1) took 0:01:59.946480


In [14]:
best1_df = pd.DataFrame([best1])
hist1_df = pd.DataFrame(hist1)

In [15]:
print(best1_df)

  Activation  Hidden Layers  Hidden Units  Learning Rate  Accuracy  F1 Score  \
0       Relu              1            64           0.01  0.689655  0.663785   

   Final Loss  Parameters      RAM MB  
0    0.151416         643  440.617188  


In [16]:
print(hist1_df)

    Activation  Hidden Layers  Hidden Units  Learning Rate  Accuracy  \
0      Sigmoid              1             2           0.10  0.394089   
1      Sigmoid              1             2           0.05  0.394089   
2      Sigmoid              1             2           0.01  0.394089   
3         Relu              1             2           0.10  0.625616   
4         Relu              1             2           0.05  0.586207   
..         ...            ...           ...            ...       ...   
265       Relu              5            64           0.05  0.512315   
266       Relu              5            64           0.01  0.394089   
267       Tanh              5            64           0.10  0.674877   
268       Tanh              5            64           0.05  0.669951   
269       Tanh              5            64           0.01  0.645320   

     F1 Score  Final Loss  Parameters      RAM MB  
0    0.222806    0.213122          23  440.617188  
1    0.222806    0.218083      

In [ ]:
grid_2 = {
    "hidden_layers": [1, 2, 3],
    "hidden_units":  [8, 16, 32, 64],
    "activation":    ["sigmoid", "relu", "tanh"],
    "epoches":       [100, 500, 1000], 
    "lr":            [0.1, 0.05, 0.01]
}

(hist2, best2), s4, e4, dur4 = measure_duration(
    grid_search,
    GRID=grid_2,
    X_train=X2_train, y_train_ohe=y2_train_ohe,
    X_test=X2_test, y_test_ohe=y2_test_ohe,
    input_size=X2_train.shape[1],
    output_size=y2_train_ohe.shape[1],
    epochs=1000, verbose=False
)

print(f"Grid Search (Dataset 2) took {dur4}")


grid_search start: 2025-05-26 22:41:32.070258
grid_search   end: 2025-05-26 22:59:07.477255
grid_search duration: 0:17:35.406997
Grid Search (Dataset 2) took 0:17:35.406997


In [18]:
best2_df = pd.DataFrame([best2])
hist2_df = pd.DataFrame(hist2)

In [19]:
print(best2_df)

  Activation  Hidden Layers  Hidden Units  Learning Rate  Accuracy  F1 Score  \
0       Tanh              2            32            0.1  0.066667  0.070294   

   Final Loss  Parameters      RAM MB  
0    0.021263      322770  379.566406  


In [20]:
print(hist2_df)

    Activation  Hidden Layers  Hidden Units  Learning Rate  Accuracy  \
0      Sigmoid              1             8           0.10  0.013333   
1      Sigmoid              1             8           0.05  0.013333   
2      Sigmoid              1             8           0.01  0.006667   
3         Relu              1             8           0.10  0.006667   
4         Relu              1             8           0.05  0.006667   
..         ...            ...           ...            ...       ...   
103       Relu              3            64           0.05  0.006667   
104       Relu              3            64           0.01  0.006667   
105       Tanh              3            64           0.10  0.046667   
106       Tanh              3            64           0.05  0.046667   
107       Tanh              3            64           0.01  0.040000   

     F1 Score  Final Loss  Parameters      RAM MB  
0    0.006753    0.027230       80466  398.363281  
1    0.007833    0.031155      